# Sentiment Analysis IndoBERT + FAISS Retrieval

Output utama:
- fine-tuned IndoBERT classifier untuk 3 label: `POSITIVE`, `NEUTRAL`, `NEGATIVE`
- metrik evaluasi pada `data/test.csv`
- prediksi untuk `data/sample_input.csv`
- top-k similar training examples dengan FAISS

## 0. Install Dependencies

Jalankan cell berikut hanya jika environment belum memiliki dependency dari `requirements.txt`.

In [1]:
# %pip install -r '/kaggle/input/datasets/tahtaaunillah/requirements/requirements.txt'

## 1. Setup

In [16]:
import inspect
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from transformers.utils import logging as transformers_logging

PROJECT_ROOT = Path('/kaggle/input/datasets/tahtaaunillah/nolimit-ds-test')

sys.path.append(str(PROJECT_ROOT))

from src.faiss_retriever import FaissRetriever, build_embeddings

MODEL_CHECKPOINT = 'indobenchmark/indobert-base-p1'
EMBEDDER_CHECKPOINT = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'

TRAIN_CSV = PROJECT_ROOT / 'data' / 'train.csv'
VAL_CSV = PROJECT_ROOT / 'data' / 'validation.csv'
TEST_CSV = PROJECT_ROOT / 'data' / 'test.csv'
SAMPLE_INPUT_CSV = PROJECT_ROOT / 'data' / 'sample_input.csv'
OUTPUT_DIR = Path('/kaggle/working/outputs/indobert-sm-sa-notebook')

LABELS = {0: 'POSITIVE', 1: 'NEUTRAL', 2: 'NEGATIVE'}
LABEL2ID = {value: key for key, value in LABELS.items()}

QUICK_RUN = False
MAX_TRAIN_SAMPLES = 300 if QUICK_RUN else 0
EPOCHS = 1 if QUICK_RUN else 3
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
SEED = 42
MAX_LENGTH = 128

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Project root:', PROJECT_ROOT)
print('Device:', device)

Project root: /kaggle/input/datasets/tahtaaunillah/nolimit-ds-test
Device: cuda


## 2. Load Data

In [17]:
def normalize_label(value):
    if pd.isna(value):
        return value
    if isinstance(value, str):
        value = value.strip()
        if value.upper() in LABEL2ID:
            return LABEL2ID[value.upper()]
    return int(value)

def load_csv_dataset(path: Path) -> Dataset:
    df = pd.read_csv(path)
    unnamed_cols = [col for col in df.columns if col.startswith('Unnamed') or col == '']
    if unnamed_cols:
        df = df.drop(columns=unnamed_cols)
    if 'text' not in df.columns:
        raise ValueError(f"{path} harus memiliki kolom 'text'")
    df['text'] = df['text'].astype(str)
    if 'label' in df.columns:
        df['label'] = df['label'].map(normalize_label).astype('int64')
    return Dataset.from_pandas(df, preserve_index=False)

train_ds = load_csv_dataset(TRAIN_CSV)
val_ds = load_csv_dataset(VAL_CSV)
test_ds = load_csv_dataset(TEST_CSV)

if MAX_TRAIN_SAMPLES:
    train_ds = train_ds.select(range(min(MAX_TRAIN_SAMPLES, len(train_ds))))

print(train_ds)
print(val_ds)
print(test_ds)
pd.DataFrame(train_ds[:5])

Dataset({
    features: ['text', 'label'],
    num_rows: 11000
})
Dataset({
    features: ['text', 'label'],
    num_rows: 1260
})
Dataset({
    features: ['text', 'label'],
    num_rows: 500
})


,text,label
0,warung ini dimiliki oleh pengusaha pabrik tahu...,0
1,mohon ulama lurus dan k212 mmbri hujjah partai...,1
2,lokasi strategis di jalan sumatera bandung . t...,0
3,betapa bahagia nya diri ini saat unboxing pake...,0
4,duh . jadi mahasiswa jangan sombong dong . kas...,2


## 3. Tokenization

In [18]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_batch(examples):
    return tokenizer(examples['text'], truncation=True, max_length=MAX_LENGTH)

train_tok = train_ds.map(tokenize_batch, batched=True)
val_tok = val_ds.map(tokenize_batch, batched=True)
test_tok = test_ds.map(tokenize_batch, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_tok[0].keys()

Map:   0%|          | 0/11000 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Map:   0%|          | 0/1260 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

dict_keys(['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])

## 4. Fine-tune IndoBERT

In [19]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_weighted': f1_score(labels, preds, average='weighted'),
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=3,
    id2label=LABELS,
    label2id=LABEL2ID,
)

def build_training_arguments(**kwargs):
    signature = inspect.signature(TrainingArguments.__init__).parameters
    if 'eval_strategy' in signature:
        kwargs['eval_strategy'] = kwargs.pop('evaluation_strategy')
    return TrainingArguments(**kwargs)

training_args = build_training_arguments(
    output_dir=str(OUTPUT_DIR),
    evaluation_strategy='epoch',
    logging_strategy='steps',
    logging_steps=25,
    save_strategy='no',
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    seed=SEED,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.evaluate()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_137/2334162811.py:37: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,0.187800,0.208604,0.930952,0.930712
2,0.108000,0.218716,0.940476,0.940265
3,0.059500,0.261040,0.941270,0.941085


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.2610403299331665,
 'eval_accuracy': 0.9412698412698413,
 'eval_f1_weighted': 0.9410849189250914,
 'eval_runtime': 6.3655,
 'eval_samples_per_second': 197.941,
 'eval_steps_per_second': 12.411,
 'epoch': 3.0}

## 5. Evaluate on Test Set

In [20]:
pred_output = trainer.predict(test_tok)
test_preds = np.argmax(pred_output.predictions, axis=-1)
test_labels = np.array(test_ds['label'], dtype=int)

print('Accuracy:', accuracy_score(test_labels, test_preds))
print('F1 weighted:', f1_score(test_labels, test_preds, average='weighted'))
print(classification_report(test_labels, test_preds, target_names=[LABELS[i] for i in range(3)]))

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Accuracy: 0.926
F1 weighted: 0.9228497570606979
              precision    recall  f1-score   support

    POSITIVE       0.93      0.96      0.94       208
     NEUTRAL       0.97      0.70      0.82        88
    NEGATIVE       0.91      0.99      0.95       204

    accuracy                           0.93       500
   macro avg       0.94      0.88      0.90       500
weighted avg       0.93      0.93      0.92       500



## 6. Predict Sample Inputs

In [ ]:
sample_df = pd.read_csv(SAMPLE_INPUT_CSV)
sample_texts = sample_df['text'].astype(str).tolist()

model.to(device)
model.eval()

sample_rows = []
for text in sample_texts:
    inputs = tokenizer(text, truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
    inputs = {key: value.to(device) for key, value in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).squeeze(0).cpu().numpy()
    pred_id = int(np.argmax(probs))
    sample_rows.append({
        'text': text,
        'predicted_label': LABELS[pred_id],
        'confidence_score': round(float(probs[pred_id]), 4),
        'positive_score': round(float(probs[0]), 4),
        'neutral_score': round(float(probs[1]), 4),
        'negative_score': round(float(probs[2]), 4),
    })

pd.DataFrame(sample_rows)


## 7. Similarity Retrieval with FAISS

In [22]:
train_df = pd.read_csv(TRAIN_CSV)
unnamed_cols = [col for col in train_df.columns if col.startswith('Unnamed') or col == '']
if unnamed_cols:
    train_df = train_df.drop(columns=unnamed_cols)

train_texts = train_df['text'].astype(str).tolist()
train_labels = train_df['label'].map(normalize_label).astype(int).tolist() if 'label' in train_df.columns else None

embedder = SentenceTransformer(EMBEDDER_CHECKPOINT)
train_embeddings = build_embeddings(train_texts, embedder, batch_size=32, normalize_embeddings=True)
retriever = FaissRetriever(train_embeddings, texts=train_texts, labels=train_labels, metric='cosine')

query_embeddings = build_embeddings(sample_texts, embedder, batch_size=8, normalize_embeddings=True)
TOP_K = 5

retrieval_rows = []
for query_index, query_text in enumerate(sample_texts):
    result = retriever.search(query_embeddings[query_index:query_index + 1], top_k=TOP_K)
    for rank, item in enumerate(retriever.get_items(result), start=1):
        label_id = item.get('label')
        retrieval_rows.append({
            'query': query_text,
            'rank': rank,
            'score': round(item['score'], 4),
            'retrieved_label': LABELS.get(int(label_id), 'N/A') if label_id is not None else 'N/A',
            'retrieved_text': item['text'],
        })

pd.DataFrame(retrieval_rows)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,query,rank,score,retrieved_label,retrieved_text
0,Doi asik bgt orangnya,1,0.8658,NEGATIVE,sombong
1,Doi asik bgt orangnya,2,0.8515,NEGATIVE,ngancem
2,Doi asik bgt orangnya,3,0.8491,NEGATIVE,nakal
3,Doi asik bgt orangnya,4,0.8416,NEGATIVE,benci
4,Doi asik bgt orangnya,5,0.8365,NEGATIVE,tumben waras nih ustaz gila dan sesat
5,"Ada pengumuman nih gaiss, besok kegiatan kanto...",1,0.5710,NEGATIVE,pesan sabtu baru sampai hari ini padahal sudah...
6,"Ada pengumuman nih gaiss, besok kegiatan kanto...",2,0.5565,NEGATIVE,menyesal tiba tiba beli kuota tadi siang
7,"Ada pengumuman nih gaiss, besok kegiatan kanto...",3,0.5558,NEGATIVE,hari ini dapat capek nya doang . pelayanan di ...
8,"Ada pengumuman nih gaiss, besok kegiatan kanto...",4,0.5552,NEGATIVE,kasihan lu parit parit di jakarta sudah mulai ...
9,"Ada pengumuman nih gaiss, besok kegiatan kanto...",5,0.5301,NEGATIVE,belum afdol kalau senin ke kantor tidak pakai ...


## 8. Save Fine-tuned Model

In [23]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
tokenizer.save_pretrained(OUTPUT_DIR)
model.save_pretrained(OUTPUT_DIR)
print('Saved model to:', OUTPUT_DIR)

Saved model to: /kaggle/working/outputs/indobert-sm-sa-notebook


In [2]:
pip install huggingface_hub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from huggingface_hub import login

login("HF_TOKEN_KAMU")

In [8]:
from huggingface_hub import HfApi

api = HfApi()

repo_id = "sovrncrypt/indobert-smsa-nolimit-ds-test"
model_folder = r"C:\Users\aunil\OneDrive\Documents\nolimit-ds-test\outputs\indobert-sm-sa-notebook"

api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    private=False,
    exist_ok=True,
)

api.upload_folder(
    folder_path=model_folder,
    repo_id=repo_id,
    repo_type="model",
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/sovrncrypt/indobert-smsa-nolimit-ds-test/commit/69f6368e51f5edd56c1846eb7b639448721feff8', commit_message='Upload folder using huggingface_hub', commit_description='', oid='69f6368e51f5edd56c1846eb7b639448721feff8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sovrncrypt/indobert-smsa-nolimit-ds-test', endpoint='https://huggingface.co', repo_type='model', repo_id='sovrncrypt/indobert-smsa-nolimit-ds-test'), pr_revision=None, pr_num=None)